In [1]:
import requests

from datetime import datetime, timedelta


In [2]:

import sys
import os

root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if root not in sys.path:
    sys.path.insert(0, root)

print(root)

/home/hilaneto/Jhan/Trabalho/python/projetos/AtualizaIndicador


#### Busca Dados IGP-M via código Atualiza-Indicador

In [ ]:

from database.conexao import conectar
from indicadores.igpm import Igpm

with conectar():
    dados_igpm = Igpm.buscar()
    for aa in dados_igpm:
        print(aa)


#### Busca Dados IGP-M via código Atualiza-Indicador

In [ ]:

from database.conexao import conectar
from indicadores.igpm import Igpm

with conectar():
    dados_igpm = Igpm.select()
    for aa in dados_igpm:
        print(aa.indice)
        print(aa.dt_referencia)


#### Busca Dados IGP-M Diretamente da API

#### Gera datas

In [3]:

# Datas -------------------------------------------------------
hoje = datetime.now()

data_inicial = (hoje - timedelta(days=365)).strftime("%d/%m/%Y")
data_final = hoje.strftime("%d/%m/%Y")

print(f"Período: {data_inicial} até {data_final}")


Período: 22/09/2025 até 22/09/2026


#### Trata e Visualiza DADOS Selic (lista de dicionário)

In [4]:

# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"

# Gera dados brutos
resposta = requests.get(url, timeout=10)
resposta.raise_for_status()
dados_brutos = resposta.json() # lista de dicionários

# Exlui duplicidade diária
dados_selic = []
valor_anterior = None
for registro in dados_brutos:
    if registro["valor"] != valor_anterior:
        dados_selic.append(registro)
        valor_anterior = registro["valor"]

# Percorrer a lista de dicionário
for selic in dados_selic:
    print(selic)


{'data': '22/09/2025', 'valor': '15.00'}
{'data': '19/03/2026', 'valor': '14.75'}
{'data': '30/04/2026', 'valor': '14.50'}
{'data': '18/06/2026', 'valor': '14.25'}
{'data': '06/08/2026', 'valor': '14.00'}
{'data': '17/09/2026', 'valor': '13.75'}


#### Insere Dados no Banco (transformando dados)

#### Visualiza DADOS Selic (lista de dicionário)

In [20]:
for selic in dados_selic:
    print(selic["data"], selic["valor"])


22/09/2025 15.00
19/03/2026 14.75
30/04/2026 14.50
18/06/2026 14.25
06/08/2026 14.00
17/09/2026 13.75


#### Transforma dados_selic e Carrega no Banco (Massivo)

In [28]:

from database.conexao import conectar
from indicadores.selic import Selic

# Transforma dados para o formato da tabela Selic
registro_selic=[]
for registro in dados_selic:
    registro_selic.append({"indice": registro["valor"],
                           "status": True,
                           "dt_referencia": datetime.strptime(registro["data"], "%d/%m/%Y").date()
                          })

with conectar():
    Selic.insert_many(registro_selic).on_conflict_ignore().execute()
    

#### Carga direta no Banco (registro por registro)

In [29]:

from database.conexao import conectar
from indicadores.selic import Selic

with conectar():
    for registro in dados_selic:
        Selic.insert(indice=registro["valor"], status=True, dt_referencia=datetime.strptime(registro["data"], "%d/%m/%Y").date()).on_conflict_ignore().execute()


#### Carga direta no Banco (Em Massa)

In [ ]:

from database.conexao import conectar
from indicadores.igpm import Igpm

with conectar():
    Igpm.insert_many(dados_igpm).on_conflict_ignore().execute()

#### Atualiza tabela IGP-M usando método do código Atualiza-Indicador

In [ ]:

from database.conexao import conectar
from indicadores.igpm import Igpm

aa = Igpm.atualizar_igpm()

print(aa)